
---

## **Part 3: Applications & Extensions**

* **Content focus:**

  * Real-world applications:

    * Economics (wages vs. education + nonparametric age effect).
    * Biostatistics/epidemiology (treatment + flexible baseline hazard).
    * Machine learning (extensions via GAMs, penalized splines).
  * Model checking and diagnostics (visualizing (g(\cdot)), residual checks).
  * Limitations and future directions (curse of dimensionality still in nonparametric part, computation).
* **Responsibilities:**

  * Collect applied case studies.
  * Run a demo on real or public dataset.
  * Present practical pros/cons vs. fully parametric or fully nonparametric.

---

In [13]:
import pandas as pd
from pygam import LinearGAM, s, f
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import scipy.sparse as sp
import plotly.express as px

In [2]:
sp.csr_matrix.A = property(lambda self: self.toarray())

In [3]:
happiness = pd.read_csv('happiness/2019.csv')

In [4]:
rename_map = {
    "GDP per capita": "gdp",
    "Social support": "support",
    "Healthy life expectancy": "lifeexp",
    "Freedom to make life choices": "freedom",
    "Generosity": "generosity",
    "Perceptions of corruption": "corruption",
    "Score": "score",
    "Country or region": "country",
    "Overall rank": "rank", 
    "Score": "score"
}
happiness = happiness.rename(columns=rename_map)

In [5]:
y_col = 'score'
X_cols = list(happiness.columns)
X_cols.remove('rank')
X_cols.remove('score')
X_cols.remove('country')
X_cols

['gdp', 'support', 'lifeexp', 'freedom', 'generosity', 'corruption']

In [6]:
happiness[X_cols]

,gdp,support,lifeexp,freedom,generosity,corruption
0,1.340,1.587,0.986,0.596,0.153,0.393
1,1.383,1.573,0.996,0.592,0.252,0.410
2,1.488,1.582,1.028,0.603,0.271,0.341
3,1.380,1.624,1.026,0.591,0.354,0.118
4,1.396,1.522,0.999,0.557,0.322,0.298
...,...,...,...,...,...,...
151,0.359,0.711,0.614,0.555,0.217,0.411
152,0.476,0.885,0.499,0.417,0.276,0.147
153,0.350,0.517,0.361,0.000,0.158,0.025
154,0.026,0.000,0.105,0.225,0.235,0.035


In [19]:
import plotly.graph_objects as go

# predictors to choose from
x_options = ["gdp","support","lifeexp","freedom","generosity","corruption"]

# create initial figure with GDP as default
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=happiness["gdp"],
    y=happiness["score"],
    mode="markers",
    marker=dict(size=8, color="blue"),
    hovertext=happiness["country"],
    name="GDP"
))

# add dropdown menu on the right
fig.update_layout(
    title="Happiness Score vs Predictor",
    xaxis_title="GDP per Capita",
    yaxis_title="Happiness Score",
    width=800,
    height=600,
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label=col,
                    method="update",
                    args=[
                        {"x": [happiness[col]],
                         "y": [happiness["score"]],
                         "hovertext": [happiness["country"]],
                         "name": col},
                        {"xaxis": {"title": col},
                         "yaxis": {"title": "Happiness Score"}}
                    ]
                )
                for col in x_options
            ],
            direction="down",
            showactive=True,
            x=1.15,         # move to right of plot
            xanchor="left",
            y=0.5,          # middle of y-axis
            yanchor="middle"
        )
    ]
)

fig.show()


In [20]:
import plotly.graph_objects as go
import statsmodels.api as sm

# predictors to test
x_options = ["gdp","support","lifeexp","freedom","generosity","corruption"]

def residuals_for_predictor(col):
    X = sm.add_constant(happiness[col])  # add intercept
    y = happiness["score"]
    model = sm.OLS(y, X).fit()
    fitted = model.predict(X)
    residuals = y - fitted
    return residuals

# make initial plot for GDP
initial_x = "gdp"
resids = residuals_for_predictor(initial_x)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=happiness[initial_x],
    y=resids,
    mode="markers",
    marker=dict(size=8, color="blue"),
    hovertext=happiness["country"],
    name=f"Residuals vs {initial_x}"
))

# horizontal line at 0
fig.add_hline(y=0, line_dash="dash", line_color="black")

# add dropdown to switch predictors
fig.update_layout(
    title="Residuals vs Predictor (Linear Fit)",
    xaxis_title=initial_x,
    yaxis_title="Residuals",
    width=800,
    height=600,
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label=col,
                    method="update",
                    args=[
                        {
                            "x": [happiness[col]],
                            "y": [residuals_for_predictor(col)],
                            "hovertext": [happiness["country"]],
                            "name": f"Residuals vs {col}"
                        },
                        {"xaxis": {"title": col}, "yaxis": {"title": "Residuals"}}
                    ]
                )
                for col in x_options
            ],
            direction="down",
            x=1.15,
            y=0.5,
            xanchor="left",
            yanchor="middle"
        )
    ]
)

fig.show()


In [21]:
import pandas as pd
import statsmodels.api as sm
from pygam import LinearGAM, s
import numpy as np

predictors = ["gdp","support","lifeexp","freedom","generosity","corruption"]

results = []

for col in predictors:
    X = sm.add_constant(happiness[col])
    y = happiness["score"]
    
    # Linear model
    lm = sm.OLS(y, X).fit()
    r2_lm = lm.rsquared
    aic_lm = lm.aic
    
    # GAM with smooth term
    X_gam = np.asarray(happiness[[col]].to_numpy(), dtype=float)
    gam = LinearGAM(s(0, n_splines=6)).fit(X_gam, y)
    r2_gam = gam.statistics_['pseudo_r2']['explained_deviance']
    gcv = gam.statistics_['GCV']  # lower = better
    
    results.append({
        "predictor": col,
        "linear_R2": r2_lm,
        "linear_AIC": aic_lm,
        "smooth_R2": r2_gam,
        "smooth_GCV": gcv
    })

df_results = pd.DataFrame(results)
df_results


,predictor,linear_R2,linear_AIC,smooth_R2,smooth_GCV
0,gdp,0.630250,323.932835,0.644609,0.464982
1,support,0.603819,334.703791,0.662258,0.440621
2,lifeexp,0.608218,332.961987,0.651260,0.455793
3,freedom,0.321196,418.703601,0.343389,0.859595
4,generosity,0.005749,478.242166,0.025671,1.270648
5,corruption,0.148697,454.027556,0.205727,1.037668


In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from pygam import LinearGAM, s
import numpy as np

x_options = ["gdp","support","lifeexp","freedom","generosity","corruption"]

def residuals_linear(col):
    X = sm.add_constant(happiness[col])
    y = happiness["score"]
    model = sm.OLS(y, X).fit()
    fitted = model.predict(X)
    return y - fitted

def residuals_gam(col):
    X = np.asarray(happiness[[col]].to_numpy(), dtype=float)
    y = np.asarray(happiness["score"].to_numpy(), dtype=float)
    gam = LinearGAM(s(0, n_splines=6)).fit(X, y)
    fitted = gam.predict(X)
    return y - fitted

initial_x = "gdp"

# --- Build subplots ---
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Linear Residuals", "Smooth (GAM) Residuals"))

# trace 0 = linear
fig.add_trace(
    go.Scatter(
        x=happiness[initial_x],
        y=residuals_linear(initial_x),
        mode="markers",
        marker=dict(size=7, color="red"),
        hovertext=happiness["country"],
        name="Linear"
    ),
    row=1, col=1
)

# trace 1 = GAM
fig.add_trace(
    go.Scatter(
        x=happiness[initial_x],
        y=residuals_gam(initial_x),
        mode="markers",
        marker=dict(size=7, color="blue"),
        hovertext=happiness["country"],
        name="GAM"
    ),
    row=1, col=2
)

# horizontal zero lines
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=2)

# --- Dropdown updates BOTH traces together ---
buttons = []
for col in x_options:
    buttons.append(
        dict(
            label=col,
            method="restyle",
            args=[
                {
                    "x": [happiness[col], happiness[col]],
                    "y": [residuals_linear(col), residuals_gam(col)],
                    "hovertext": [happiness["country"], happiness["country"]],
                },
                [0, 1]  # update both traces
            ]
        )
    )

fig.update_layout(
    title=f"Residual Comparison: {initial_x}",
    width=1000,
    height=500,
    showlegend=False,
    updatemenus=[
        dict(
            type="dropdown",
            buttons=buttons,
            x=0.5,
            xanchor="center",
            y=1.15,
            yanchor="top"
        )
    ]
)

fig.show()


In [35]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from pygam import LinearGAM, s
import numpy as np

x_options = ["gdp","support","lifeexp","freedom","generosity","corruption"]

def fitted_linear(col):
    X = sm.add_constant(happiness[col])
    y = happiness["score"]
    model = sm.OLS(y, X).fit()
    x_sorted = np.linspace(happiness[col].min(), happiness[col].max(), 100)
    X_pred = sm.add_constant(x_sorted)
    y_pred = model.predict(X_pred)
    return x_sorted, y_pred

def fitted_gam(col):
    X = np.asarray(happiness[[col]].to_numpy(), dtype=float)
    y = np.asarray(happiness["score"].to_numpy(), dtype=float)
    gam = LinearGAM(s(0, n_splines=6)).fit(X, y)
    x_sorted = np.linspace(X.min(), X.max(), 100)[:, None]
    y_pred = gam.predict(x_sorted)
    return x_sorted.flatten(), y_pred

initial_x = "gdp"

# --- Build subplots ---
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Linear Fit", "Smooth (GAM) Fit"))

# trace 0 = linear line
x_lin, y_lin = fitted_linear(initial_x)
fig.add_trace(go.Scatter(x=x_lin, y=y_lin, mode="lines",
                         line=dict(color="red"), name="Linear"),
              row=1, col=1)

# trace 1 = GAM line
x_gam, y_gam = fitted_gam(initial_x)
fig.add_trace(go.Scatter(x=x_gam, y=y_gam, mode="lines",
                         line=dict(color="blue"), name="GAM"),
              row=1, col=2)

# trace 2 = scatter (linear panel)
fig.add_trace(go.Scatter(x=happiness[initial_x], y=happiness["score"],
                         mode="markers", marker=dict(size=6, color="gray", opacity=0.6),
                         showlegend=False), row=1, col=1)

# trace 3 = scatter (GAM panel)
fig.add_trace(go.Scatter(x=happiness[initial_x], y=happiness["score"],
                         mode="markers", marker=dict(size=6, color="gray", opacity=0.6),
                         showlegend=False), row=1, col=2)

# --- Dropdown updates all 4 traces ---
buttons = []
for col in x_options:
    x_lin, y_lin = fitted_linear(col)
    x_gam, y_gam = fitted_gam(col)
    buttons.append(
        dict(
            label=col,
            method="update",
            args=[
                {
                    "x": [x_lin, x_gam, happiness[col], happiness[col]],
                    "y": [y_lin, y_gam, happiness["score"], happiness["score"]],
                },
                [0, 1, 2, 3]
            ]
        )
    )

fig.update_layout(
    title=f"Model Fit Comparison: {initial_x}",
    width=1000,
    height=500,
    updatemenus=[dict(type="dropdown", buttons=buttons,
                      x=0.5, y=1.15, xanchor="center", yanchor="top")]
)

fig.show()
